# RAGAS를 활용한 GraphRAG (Text2Cypher) 평가

Neo4j Sandbox의 **영화 추천 데이터셋(Recommendations)** 을 활용하여  
Text2Cypher 기반 GraphRAG 파이프라인을 구축하고, **RAGAS** 라이브러리로 5가지 지표를 평가합니다.

### 평가 지표
| 지표 | 설명 |
|------|------|
| **Faithfulness** | 답변이 검색된 context에 충실한지 |
| **Answer Relevancy** | 답변이 질문과 관련 있는지 |
| **Context Recall** | 정답(ground truth)을 뒷받침하는 context가 얼마나 검색되었는지 |
| **Context Precision** | 검색된 context 중 관련 있는 것의 비율 |
| **Answer Accuracy** | 답변이 정답(ground truth)과 얼마나 일치하는지 |

### Naive RAG vs GraphRAG 차이점
- Naive RAG: 벡터 검색으로 문서 chunk를 가져옴 → context가 이미 텍스트
- **GraphRAG**: Cypher 쿼리로 그래프 DB에서 구조화된 데이터를 가져옴 → **context를 텍스트로 변환하는 정제 과정이 필요**

## 1. 환경 설정

In [1]:
import os
import json
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

True

### RAGAS 단일 샘플 평가 예시 (Collections API)

아래 예시들은 **단일 샘플 평가**를 위한 Collections API 사용법입니다.  
- 사용: `ragas.metrics.collections`에서 import
- 평가: `.ascore()` 메서드로 개별 샘플 평가
- 특징: OpenAI 클라이언트 직접 사용, 간결한 인터페이스

**참고**: 이후 배치 평가(`evaluate()` 함수)에서는 Legacy API를 사용합니다.

- Faithfulness

In [2]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import Faithfulness

# Setup LLM
client = AsyncOpenAI()
llm = llm_factory("gpt-4o-mini", client=client)

# Create metric
scorer = Faithfulness(llm=llm)

# Evaluate
result = await scorer.ascore(
    user_input="아인슈타인은 몇년도에 태어났나요?",
    response="아인슈타인은 1879년에 태어났습니다.",
    retrieved_contexts=[
        "알베르트 아인슈타인은 1879년 3월 14일에 독일 울름에서 태어났습니다."
    ]
)
print(f"Faithfulness Score: {result.value}")

Faithfulness Score: 1.0


- AnswerRelevancy

In [3]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory
from ragas.metrics.collections import AnswerRelevancy

# Setup LLM and embeddings
client = AsyncOpenAI()
llm = llm_factory("gpt-4o-mini", client=client)
embeddings = embedding_factory("openai", model="text-embedding-3-small", client=client)

# Create metric
scorer = AnswerRelevancy(llm=llm, embeddings=embeddings)

# Evaluate
result = await scorer.ascore(
    user_input="2024년 노벨문학상 수상자는 누구인가요?",
    response="한강(Han Kang)입니다."
)
print(f"Answer Relevancy Score: {result.value}")

result = await scorer.ascore(
    user_input="AI란 무엇인가요?",
    response="AI는 인공지능(Artificial Intelligence)의 약자로, 기계가 인간처럼 학습하고 문제를 해결할 수 있는 능력을 의미합니다."
)
print(f"Answer Relevancy Score: {result.value}")

Answer Relevancy Score: 0.0971138389105794
Answer Relevancy Score: 0.4770022403971442


- ContextRecall

In [4]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import ContextRecall

# Setup LLM
client = AsyncOpenAI()
llm = llm_factory("gpt-4o-mini", client=client)

# Create metric
scorer = ContextRecall(llm=llm)

# Evaluate
result = await scorer.ascore(
    user_input="에펠탑이 어디에 있나요?",
    reference="에펠탑은 파리에 있습니다.",
    retrieved_contexts=[
        "에펠탑은 높습니다.",
        "파리는 에펠탑으로 유명합니다.",
    ]
)
print(f"Context Recall Score: {result.value}")

Context Recall Score: 1.0


- ContextPrecision

In [5]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import ContextPrecision

# Setup LLM
client = AsyncOpenAI()
llm = llm_factory("gpt-4o-mini", client=client)

# Create metric
scorer = ContextPrecision(llm=llm)

# Evaluate
result = await scorer.ascore(
    user_input="에펠탑이 어디에 있나요?",
    reference="에펠탑은 파리에 있습니다.",
    retrieved_contexts=[
        "에펠탑은 높습니다.",
        "파리는 에펠탑으로 유명합니다.",
    ]
)
print(f"Context Precision Score: {result.value}")

Context Precision Score: 0.49999999995


- AnswerAccuracy

In [6]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import AnswerAccuracy

# Setup LLM
client = AsyncOpenAI()
llm = llm_factory("gpt-4o-mini", client=client)

# Create metric
scorer = AnswerAccuracy(llm=llm)

# Evaluate
result = await scorer.ascore(
    user_input="아인슈타인은 언제 태어났나요?",
    response="알베르트 아인슈타인은 1879년에 태어났습니다.",
    reference="알베르트 아인슈타인은 1879년에 태어났습니다."
)
print(f"Answer Accuracy Score: {result.value}")

Answer Accuracy Score: 1.0


## 2. Neo4j 연결 및 스키마 확인

In [61]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USERNAME"), os.getenv("NEO4J_PASSWORD"))
)

driver.verify_connectivity()
print(f"Neo4j 연결 성공: {os.getenv('NEO4J_URI')}")

Neo4j 연결 성공: bolt://18.209.59.92:7687


In [62]:
# 스키마 확인: 노드 라벨, 관계 타입, 속성 등

# 노드 라벨
records, _, _ = driver.execute_query("CALL db.labels()")
labels = [r[0] for r in records]
print("노드 라벨:", labels)

# 관계 타입
records, _, _ = driver.execute_query("CALL db.relationshipTypes()")
rel_types = [r[0] for r in records]
print("관계 타입:", rel_types)

# 노드별 속성 확인
print("\n--- 노드별 속성 ---")
for label in labels:
    records, _, _ = driver.execute_query(
        f"MATCH (n:`{label}`) RETURN keys(n) AS keys LIMIT 1"
    )
    if records:
        print(f"{label}: {records[0]['keys']}")

# 관계 패턴 확인
print("\n--- 관계 패턴 ---")
records, _, _ = driver.execute_query("CALL db.schema.visualization()")
if records:
    for rel in records[0]["relationships"]:
        print(f"(:{rel.start_node['name']})-[:{rel.type}]->(:{rel.end_node['name']})")

노드 라벨: ['Movie', 'Genre', 'User', 'Actor', 'Director', 'Person']
관계 타입: ['IN_GENRE', 'RATED', 'ACTED_IN', 'DIRECTED']

--- 노드별 속성 ---
Movie: ['url', 'runtime', 'revenue', 'budget', 'imdbRating', 'released', 'countries', 'languages', 'plot', 'imdbVotes', 'imdbId', 'year', 'poster', 'movieId', 'tmdbId', 'title']
Genre: ['name']
User: ['userId', 'name']
Actor: ['bornIn', 'born', 'died', 'tmdbId', 'imdbId', 'name', 'url']
Director: ['url', 'bornIn', 'bio', 'died', 'born', 'imdbId', 'name', 'poster', 'tmdbId']
Person: ['bornIn', 'born', 'died', 'tmdbId', 'imdbId', 'name', 'url']

--- 관계 패턴 ---
(:Actor)-[:ACTED_IN]->(:Movie)
(:Person)-[:ACTED_IN]->(:Movie)
(:Director)-[:ACTED_IN]->(:Movie)
(:User)-[:RATED]->(:Movie)
(:Movie)-[:IN_GENRE]->(:Genre)
(:Person)-[:DIRECTED]->(:Movie)
(:Actor)-[:DIRECTED]->(:Movie)
(:Director)-[:DIRECTED]->(:Movie)


## 3. 영화 추천 데이터셋 스키마 정의

Neo4j Sandbox Recommendations 데이터셋의 스키마를 Text2CypherRetriever가 이해할 수 있는 형태로 정의합니다.

In [63]:
# Neo4j Sandbox Recommendations 스키마 (자동 추출)
def get_neo4j_schema(driver):
    """Neo4j DB에서 스키마를 자동으로 추출하여 텍스트로 변환"""
    lines = []

    # 노드 속성 추출
    lines.append("Node properties:")
    records, _, _ = driver.execute_query("CALL db.labels()")
    labels = [r[0] for r in records]
    for label in labels:
        records, _, _ = driver.execute_query(
            f"MATCH (n:`{label}`) RETURN keys(n) AS keys LIMIT 1"
        )
        if records and records[0]["keys"]:
            # 속성 타입 확인
            props = []
            sample_records, _, _ = driver.execute_query(
                f"MATCH (n:`{label}`) RETURN n LIMIT 1"
            )
            if sample_records:
                node = sample_records[0]["n"]
                for key in records[0]["keys"]:
                    val = node.get(key)
                    if isinstance(val, int):
                        dtype = "INTEGER"
                    elif isinstance(val, float):
                        dtype = "FLOAT"
                    elif isinstance(val, list):
                        dtype = "LIST"
                    else:
                        dtype = "STRING"
                    props.append(f"{key}: {dtype}")
            lines.append(f"{label} {{{', '.join(props)}}}")

    # 관계 속성 추출
    lines.append("\nRelationship properties:")
    records, _, _ = driver.execute_query("CALL db.relationshipTypes()")
    rel_types = [r[0] for r in records]
    for rel_type in rel_types:
        records, _, _ = driver.execute_query(
            f"MATCH ()-[r:`{rel_type}`]->() RETURN keys(r) AS keys LIMIT 1"
        )
        if records and records[0]["keys"]:
            props = []
            sample_records, _, _ = driver.execute_query(
                f"MATCH ()-[r:`{rel_type}`]->() RETURN r LIMIT 1"
            )
            if sample_records:
                rel = sample_records[0]["r"]
                for key in records[0]["keys"]:
                    val = rel.get(key)
                    if isinstance(val, int):
                        dtype = "INTEGER"
                    elif isinstance(val, float):
                        dtype = "FLOAT"
                    elif isinstance(val, list):
                        dtype = "LIST"
                    else:
                        dtype = "STRING"
                    props.append(f"{key}: {dtype}")
            lines.append(f"{rel_type} {{{', '.join(props)}}}")

    # 관계 패턴 추출
    lines.append("\nThe relationships:")
    for rel_type in rel_types:
        records, _, _ = driver.execute_query(f"""
            MATCH (a)-[r:`{rel_type}`]->(b)
            RETURN DISTINCT labels(a)[0] AS src, labels(b)[0] AS tgt
            LIMIT 5
        """)
        for rec in records:
            lines.append(f"(:{rec['src']})-[:{rel_type}]->(:{rec['tgt']})")

    return "\n".join(lines)

neo4j_schema = get_neo4j_schema(driver)
print(neo4j_schema)

Node properties:
Movie {url: STRING, runtime: INTEGER, revenue: INTEGER, budget: INTEGER, imdbRating: FLOAT, released: STRING, countries: LIST, languages: LIST, plot: STRING, imdbVotes: INTEGER, imdbId: STRING, year: INTEGER, poster: STRING, movieId: STRING, tmdbId: STRING, title: STRING}
Genre {name: STRING}
User {userId: STRING, name: STRING}
Actor {bornIn: STRING, born: STRING, died: STRING, tmdbId: STRING, imdbId: STRING, name: STRING, url: STRING}
Director {url: STRING, bornIn: STRING, bio: STRING, died: STRING, born: STRING, imdbId: STRING, name: STRING, poster: STRING, tmdbId: STRING}
Person {bornIn: STRING, born: STRING, died: STRING, tmdbId: STRING, imdbId: STRING, name: STRING, url: STRING}

Relationship properties:
RATED {rating: FLOAT, timestamp: INTEGER}
ACTED_IN {role: STRING}

The relationships:
(:Movie)-[:IN_GENRE]->(:Genre)
(:User)-[:RATED]->(:Movie)
(:Actor)-[:ACTED_IN]->(:Movie)
(:Actor)-[:DIRECTED]->(:Movie)
(:Director)-[:DIRECTED]->(:Movie)


## 4. Text2Cypher GraphRAG 파이프라인 구축

In [64]:
from neo4j_graphrag.retrievers import Text2CypherRetriever
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.generation import RagTemplate, GraphRAG

llm = OpenAILLM(
    model_name="gpt-4o",
    model_params={"temperature": 0}
)

examples = [
    "USER INPUT: '데이터베이스에 영화가 몇건 들어있나요?' QUERY: MATCH (m:Movie) RETURN count(m) AS movie_count",
    "USER INPUT: '누가 매트릭스에 출연했나요?' QUERY: MATCH (m:Movie) WHERE m.title CONTAINS 'Matrix' MATCH (p:Person)-[:ACTED_IN]->(m) RETURN m.title AS movie, p.name AS actor ORDER BY m.title, p.name",
    "USER INPUT: '톰 행크스가 출연한 영화는 무엇인가요?' QUERY: MATCH (p:Person)-[:ACTED_IN]->(m:Movie) WHERE p.name CONTAINS 'Hanks' RETURN m.title AS movie, m.year AS year ORDER BY m.year",
    "USER INPUT: '매트릭스를 감독한 사람은 누구인가요?' QUERY: MATCH (m:Movie) WHERE m.title CONTAINS 'Matrix' MATCH (p:Person)-[:DIRECTED]->(m) RETURN m.title AS movie, p.name AS director ORDER BY m.title",
    "USER INPUT: '토이 스토리 영화의 장르는 무엇인가요?' QUERY: MATCH (m:Movie) WHERE m.title CONTAINS 'Toy Story' MATCH (m)-[:IN_GENRE]->(g:Genre) RETURN DISTINCT m.title AS movie, g.name AS genre ORDER BY m.title, g.name",
    "USER INPUT: '평점이 높은 영화는 무엇인가요?' QUERY: MATCH (m:Movie) WHERE m.imdbRating IS NOT NULL RETURN m.title AS title, m.imdbRating AS rating ORDER BY m.imdbRating DESC LIMIT 10",
    "USER INPUT: '키아누 리브스가 출연한 영화는 몇 편인가요?' QUERY: MATCH (p:Person)-[:ACTED_IN]->(m:Movie) WHERE p.name CONTAINS 'Reeves' RETURN count(DISTINCT m) AS movie_count",
    "USER INPUT: '1999년에 개봉한 영화는 무엇인가요?' QUERY: MATCH (m:Movie) WHERE m.year = 1999 RETURN m.title AS title ORDER BY m.title LIMIT 3",
    "USER INPUT: '매트릭스 시리즈에 모두 출연한 배우는 누구인가요?' QUERY: MATCH (m1:Movie) WHERE m1.title CONTAINS 'Matrix' AND NOT m1.title CONTAINS 'Reloaded' AND NOT m1.title CONTAINS 'Revolutions' MATCH (m2:Movie) WHERE m2.title CONTAINS 'Matrix Reloaded' MATCH (p:Person)-[:ACTED_IN]->(m1), (p)-[:ACTED_IN]->(m2) RETURN DISTINCT p.name AS actor ORDER BY p.name",
    "USER INPUT: '인셉션과 장르가 비슷한 영화를 추천해 주세요' QUERY: MATCH (m:Movie) WHERE m.title CONTAINS 'Inception' WITH m LIMIT 1 MATCH (m)-[:IN_GENRE]->(g:Genre)<-[:IN_GENRE]-(rec:Movie) WHERE rec <> m RETURN rec.title AS recommended, collect(g.name) AS shared_genres, count(g) AS genre_overlap ORDER BY genre_overlap DESC LIMIT 10",
]

In [65]:
# Text2Cypher 커스텀 프롬프트 정의
# Recommendations sandbox에서 영화 제목이 "Matrix, The" 등으로 저장되어 있을 수 있으므로
# CONTAINS를 사용한 유연한 매칭을 지시하되, 모든 매칭 결과를 반환
custom_prompt = """Task: 사용자의 입력으로부터 Neo4j 그래프 데이터베이스를 쿼리하기 위한 Cypher 문을 생성하세요.

<schema>
{schema}
</schema>

<Examples (optional)>
{examples}
</Examples>

Important Rules:
1. 영화 제목은 이 데이터베이스에서 비표준 형식으로 저장될 수 있습니다 (예: "Matrix, The" 대신 "The Matrix").
   - 영화 제목으로 검색할 때는 정확한 일치 대신 특징적인 키워드를 사용하여 CONTAINS를 사용하세요.
   - 예: 정확한 제목 대신 WHERE m.title CONTAINS 'Matrix'를 사용하세요.
   - 질문이 명확히 단일 결과를 요구하지 않는 한, 매칭되는 모든 영화를 반환하세요.
   - 여러 영화가 매칭될 수 있으므로 결과에 영화 제목을 포함하세요 (예: RETURN m.title, p.name).
2. 사람 이름도 다양할 수 있으므로 적절할 때 이름 일치에 CONTAINS를 사용하는 것이 좋습니다.
3. 중복을 피하기 위해 집계 시에는 DISTINCT를 사용하세요.
4. LIMIT은 다음 경우에만 사용하세요:
   - 질문이 명시적으로 "상위 N개" 또는 "첫 N개"를 요구하는 경우
   - 추천 또는 집계 쿼리에서 너무 많은 결과를 피하기 위한 경우
5. 스키마에 포함되지 않은 속성이나 관계를 사용하지 마세요.
6. Cypher 문 외에 추가적인 텍스트나 백틱 ```를 포함하지 말고, 오직 생성된 Cypher 문만 응답하세요.

Input:
{query_text}

Cypher query:
"""

# Text2Cypher Retriever 생성
retriever = Text2CypherRetriever(
    driver=driver,
    llm=llm,
    neo4j_schema=neo4j_schema,
    examples=examples,
    custom_prompt=custom_prompt,
)

# RAG 프롬프트 템플릿 - Answer Relevancy 개선 버전
prompt_template = RagTemplate(
    template="""
당신은 영화 데이터베이스 전문가입니다. 질문에 직접적이고 간결하게 답변하세요.

<retrieval_result>
{context}
</retrieval_result>

<user_query>
{query_text}
</user_query>

<guidelines>
CRITICAL - 답변 작성 규칙:
1. 질문에 직접 답하는 핵심 정보만 포함하세요.
2. 여러 영화가 검색된 경우, 영화별로 명확히 구분하여 나열하세요.
3. 검색된 모든 데이터를 빠짐없이 포함하되, 간결하게 작성하세요.
4. 금지 사항:
   - "검색된 데이터에 따르면...", "이 정보는..." 같은 서론 금지
   - "데이터에 포함되지 않음", "추가 정보 필요" 같은 부가 설명 금지
   - context에 없는 정보나 배경 지식 추가 금지
   - 불필요한 강조나 형식적 문구 금지
5. 간결하고 직접적인 답변이 최우선입니다.
</guidelines>

Answer:
""",
    expected_inputs=["context", "query_text"]
)

# GraphRAG 파이프라인 조립
rag = GraphRAG(retriever=retriever, llm=llm, prompt_template=prompt_template)

In [66]:
test_response = rag.search(
    query_text="매트릭스 영화에 누가 출연했어?",
    return_context=True
)

print("생성된 Cypher:")
print(test_response.retriever_result.metadata["cypher"])
print("\n검색된 context:")
for item in test_response.retriever_result.items:
    print(f"  - {item.content}")
print("\n답변:")
print(test_response.answer)

생성된 Cypher:
MATCH (m:Movie) WHERE m.title CONTAINS 'Matrix' MATCH (p:Person)-[:ACTED_IN]->(m) RETURN m.title AS movie, p.name AS actor ORDER BY m.title, p.name

검색된 context:
  - <Record movie='Matrix Reloaded, The' actor='Andy Arness'>
  - <Record movie='Matrix Reloaded, The' actor='Carrie-Anne Moss'>
  - <Record movie='Matrix Reloaded, The' actor='Christine Anu'>
  - <Record movie='Matrix Reloaded, The' actor='Keanu Reeves'>
  - <Record movie='Matrix Revolutions, The' actor='Helmut Bakaitis'>
  - <Record movie='Matrix Revolutions, The' actor='Kate Beahan'>
  - <Record movie='Matrix Revolutions, The' actor='Keanu Reeves'>
  - <Record movie='Matrix Revolutions, The' actor='Mary Alice'>
  - <Record movie='Matrix, The' actor='Carrie-Anne Moss'>
  - <Record movie='Matrix, The' actor='Hugo Weaving'>
  - <Record movie='Matrix, The' actor='Keanu Reeves'>
  - <Record movie='Matrix, The' actor='Laurence Fishburne'>

답변:
- 매트릭스 (The Matrix): Keanu Reeves, Carrie-Anne Moss, Hugo Weaving, Laurence

## 5. 평가용 테스트 데이터셋 정의

GraphRAG 평가를 위해 **질문(question)** 과 **정답(ground_truth)** 을 수동으로 정의합니다.  
Neo4j Sandbox Recommendations 데이터셋 기준으로 작성되었습니다.

In [67]:
# 테스트셋: 다양한 난이도와 유형의 질문
# ground_truth는 실제 Neo4j 데이터 기반으로 작성
# NOTE: CONTAINS로 매칭되는 모든 영화를 반환하며, 영화 제목을 포함하여 구분 가능하도록 함

ground_truth_queries = {
    "q1": "MATCH (m:Movie) WHERE m.title CONTAINS 'Matrix' MATCH (p:Person)-[:ACTED_IN]->(m) RETURN m.title AS movie, p.name AS actor ORDER BY m.title, p.name",
    "q2": "MATCH (m:Movie) WHERE m.title CONTAINS 'Matrix' MATCH (p:Person)-[:DIRECTED]->(m) RETURN m.title AS movie, p.name AS director ORDER BY m.title, p.name",
    "q3": "MATCH (p:Person)-[:ACTED_IN]->(m:Movie) WHERE p.name CONTAINS 'Tom Hanks' RETURN count(DISTINCT m) AS cnt",
    "q4": "MATCH (m:Movie) WHERE m.imdbRating IS NOT NULL RETURN m.title, m.imdbRating ORDER BY m.imdbRating DESC LIMIT 5",
    "q5": "MATCH (m:Movie) WHERE m.title CONTAINS 'Forrest Gump' MATCH (m)-[:IN_GENRE]->(g:Genre) RETURN DISTINCT m.title AS movie, g.name AS genre ORDER BY m.title, g.name",
    "q6": "MATCH (m1:Movie) WHERE m1.title CONTAINS 'Matrix' AND NOT m1.title CONTAINS 'Reloaded' AND NOT m1.title CONTAINS 'Revolutions' MATCH (m2:Movie) WHERE m2.title CONTAINS 'Matrix Reloaded' MATCH (p:Person)-[:ACTED_IN]->(m1), (p)-[:ACTED_IN]->(m2) RETURN DISTINCT p.name AS actor ORDER BY p.name",
    "q7": "MATCH (p:Person)-[:DIRECTED]->(m:Movie) RETURN p.name, count(DISTINCT m) AS cnt ORDER BY cnt DESC LIMIT 1",
    "q8": "MATCH (m:Movie) WHERE m.year = 1999 RETURN m.title ORDER BY m.title LIMIT 5",
    "q9": "MATCH (m:Movie) WHERE m.title CONTAINS 'Pulp Fiction' MATCH (p:Person)-[:ACTED_IN]->(m) RETURN m.title AS movie, p.name AS actor ORDER BY m.title, p.name",
    "q10": "MATCH (m:Movie) WHERE m.title CONTAINS 'Inception' WITH m LIMIT 1 MATCH (m)-[:IN_GENRE]->(g:Genre)<-[:IN_GENRE]-(rec:Movie) WHERE rec <> m RETURN DISTINCT rec.title, count(g) AS overlap ORDER BY overlap DESC LIMIT 5",
}

gt_results = {}
for key, query in ground_truth_queries.items():
    try:
        records, _, _ = driver.execute_query(query)
        result = [list(r.values()) for r in records]
        gt_results[key] = result
        print(f"{key}: {result}{''}")
    except Exception as e:
        print(f"{key}: 오류 - {e}")
        gt_results[key] = []

q1: [['Matrix Reloaded, The', 'Andy Arness'], ['Matrix Reloaded, The', 'Carrie-Anne Moss'], ['Matrix Reloaded, The', 'Christine Anu'], ['Matrix Reloaded, The', 'Keanu Reeves'], ['Matrix Revolutions, The', 'Helmut Bakaitis'], ['Matrix Revolutions, The', 'Kate Beahan'], ['Matrix Revolutions, The', 'Keanu Reeves'], ['Matrix Revolutions, The', 'Mary Alice'], ['Matrix, The', 'Carrie-Anne Moss'], ['Matrix, The', 'Hugo Weaving'], ['Matrix, The', 'Keanu Reeves'], ['Matrix, The', 'Laurence Fishburne']]
q2: [['Matrix Reloaded, The', ' Lilly Wachowski'], ['Matrix Reloaded, The', 'Lana Wachowski'], ['Matrix Revolutions, The', ' Lilly Wachowski'], ['Matrix Revolutions, The', 'Lana Wachowski'], ['Matrix, The', ' Lilly Wachowski'], ['Matrix, The', 'Lana Wachowski']]
q3: [[38]]
q4: [['Band of Brothers', 9.6], ['Civil War, The', 9.5], ['Cosmos', 9.3], ['Shawshank Redemption, The', 9.3], ['Decalogue, The (Dekalog)', 9.2]]
q5: [['Forrest Gump', 'Comedy'], ['Forrest Gump', 'Drama'], ['Forrest Gump', 'Roma

In [68]:
gt_results

{'q1': [['Matrix Reloaded, The', 'Andy Arness'],
  ['Matrix Reloaded, The', 'Carrie-Anne Moss'],
  ['Matrix Reloaded, The', 'Christine Anu'],
  ['Matrix Reloaded, The', 'Keanu Reeves'],
  ['Matrix Revolutions, The', 'Helmut Bakaitis'],
  ['Matrix Revolutions, The', 'Kate Beahan'],
  ['Matrix Revolutions, The', 'Keanu Reeves'],
  ['Matrix Revolutions, The', 'Mary Alice'],
  ['Matrix, The', 'Carrie-Anne Moss'],
  ['Matrix, The', 'Hugo Weaving'],
  ['Matrix, The', 'Keanu Reeves'],
  ['Matrix, The', 'Laurence Fishburne']],
 'q2': [['Matrix Reloaded, The', ' Lilly Wachowski'],
  ['Matrix Reloaded, The', 'Lana Wachowski'],
  ['Matrix Revolutions, The', ' Lilly Wachowski'],
  ['Matrix Revolutions, The', 'Lana Wachowski'],
  ['Matrix, The', ' Lilly Wachowski'],
  ['Matrix, The', 'Lana Wachowski']],
 'q3': [[38]],
 'q4': [['Band of Brothers', 9.6],
  ['Civil War, The', 9.5],
  ['Cosmos', 9.3],
  ['Shawshank Redemption, The', 9.3],
  ['Decalogue, The (Dekalog)', 9.2]],
 'q5': [['Forrest Gump', '

In [69]:
def format_gt_result(values, template=""):
    """DB 쿼리 결과를 ground truth 텍스트로 변환"""
    if not values:
        return "No results found."
    # 단일 컬럼 결과
    if len(values[0]) == 1:
        items = [str(v[0]) for v in values]
        return ", ".join(items)
    # 다중 컬럼 결과
    rows = []
    for row in values:
        rows.append(" - ".join(str(v) for v in row))
    return "; ".join(rows)


# 평가용 데이터셋 정의
test_data = [
    {
        "question": "매트릭스에서 누가 연기했나요?",
        "ground_truth": f"매트릭스 시리즈 영화들과 출연 배우는 다음과 같습니다: {format_gt_result(gt_results.get('q1', []))}",
    },
    {
        "question": "매트릭스를 누가 감독했나요?",
        "ground_truth": f"매트릭스 시리즈 영화들과 감독은 다음과 같습니다: {format_gt_result(gt_results.get('q2', []))}",
    },
    {
        "question": "톰 행크스는 몇 편의 영화에 출연했나요?",
        "ground_truth": f"톰 행크스는 {format_gt_result(gt_results.get('q3', []))}편의 영화에 출연했습니다.",
    },
    {
        "question": "가장 높은 평점을 받은 상위 5개의 영화는 무엇인가요?",
        "ground_truth": f"가장 높은 평점을 받은 상위 5개의 영화는 다음과 같습니다: {format_gt_result(gt_results.get('q4', []))}",
    },
    {
        "question": "포레스트 검프는 어떤 장르에 속하나요?",
        "ground_truth": f"포레스트 검프 영화와 장르는 다음과 같습니다: {format_gt_result(gt_results.get('q5', []))}",
    },
    {
        "question": "매트릭스와 매트릭스 리로디드에 모두 출연한 배우는 누구인가요?",
        "ground_truth": f"매트릭스와 매트릭스 리로디드에 모두 출연한 배우는 다음과 같습니다: {format_gt_result(gt_results.get('q6', []))}",
    },
    {
        "question": "가장 많은 영화를 감독한 감독은 누구인가요?",
        "ground_truth": f"가장 많은 영화를 감독한 감독은 {format_gt_result(gt_results.get('q7', []))}입니다.",
    },
    {
        "question": "1999년에 개봉한 영화는 무엇인가요?",
        "ground_truth": f"1999년에 개봉한 영화는 다음과 같습니다: {format_gt_result(gt_results.get('q8', []))}",
    },
    {
        "question": "펄프 픽션에 출연한 배우는 누구인가요?",
        "ground_truth": f"펄프 픽션 영화와 출연 배우는 다음과 같습니다: {format_gt_result(gt_results.get('q9', []))}",
    },
    {
        "question": "인셉션과 장르가 비슷한 영화를 추천해주세요.",
        "ground_truth": f"인셉션과 장르가 비슷한 영화는 다음과 같습니다: {format_gt_result(gt_results.get('q10', []))}",
    },
]

print(f"테스트 데이터셋: {len(test_data)}개 질문")
for i, d in enumerate(test_data, 1):
    print(f"  Q{i}: {d['question']}")
    print(f"  GT: {d['ground_truth'][:100]}...\n")

테스트 데이터셋: 10개 질문
  Q1: 매트릭스에서 누가 연기했나요?
  GT: 매트릭스 시리즈 영화들과 출연 배우는 다음과 같습니다: Matrix Reloaded, The - Andy Arness; Matrix Reloaded, The - Carrie-Ann...

  Q2: 매트릭스를 누가 감독했나요?
  GT: 매트릭스 시리즈 영화들과 감독은 다음과 같습니다: Matrix Reloaded, The -  Lilly Wachowski; Matrix Reloaded, The - Lana Wac...

  Q3: 톰 행크스는 몇 편의 영화에 출연했나요?
  GT: 톰 행크스는 38편의 영화에 출연했습니다....

  Q4: 가장 높은 평점을 받은 상위 5개의 영화는 무엇인가요?
  GT: 가장 높은 평점을 받은 상위 5개의 영화는 다음과 같습니다: Band of Brothers - 9.6; Civil War, The - 9.5; Cosmos - 9.3; Shawsh...

  Q5: 포레스트 검프는 어떤 장르에 속하나요?
  GT: 포레스트 검프 영화와 장르는 다음과 같습니다: Forrest Gump - Comedy; Forrest Gump - Drama; Forrest Gump - Romance; Forre...

  Q6: 매트릭스와 매트릭스 리로디드에 모두 출연한 배우는 누구인가요?
  GT: 매트릭스와 매트릭스 리로디드에 모두 출연한 배우는 다음과 같습니다: Carrie-Anne Moss, Keanu Reeves...

  Q7: 가장 많은 영화를 감독한 감독은 누구인가요?
  GT: 가장 많은 영화를 감독한 감독은 Woody Allen - 42입니다....

  Q8: 1999년에 개봉한 영화는 무엇인가요?
  GT: 1999년에 개봉한 영화는 다음과 같습니다: 10 Things I Hate About You, 13th Warrior, The, 200 Cigarettes, 24 Hour Woma...

  Q9: 펄프 픽션에 출연한 배우는 누구인가요?

## 6. GraphRAG 실행 및 응답 수집

각 질문에 대해 GraphRAG를 실행하고, **답변(answer)** 과 **검색된 context** 를 함께 수집합니다.  
GraphRAG의 context는 Cypher 쿼리 결과(구조화 데이터)이므로 **텍스트로 변환하는 정제 과정**이 필요합니다.

In [70]:
def convert_graph_context_to_text(retriever_result):
    context_pieces = []

    # 생성된 Cypher 쿼리 정보 포함
    cypher = retriever_result.metadata.get("cypher", "")
    if cypher:
        context_pieces.append(f"Executed Cypher query: {cypher}")

    # 검색된 각 레코드를 텍스트로 변환
    for item in retriever_result.items:
        content = item.content
        # item.content는 이미 문자열인 경우가 많음
        if isinstance(content, str):
            context_pieces.append(content)
        elif isinstance(content, dict):
            # dict인 경우 key-value 쌍을 텍스트로 변환
            parts = [f"{k}: {v}" for k, v in content.items()]
            context_pieces.append(", ".join(parts))
        else:
            context_pieces.append(str(content))

    return "\n".join(context_pieces)

In [71]:
# 모든 질문에 대해 GraphRAG 실행
results = []

for i, data in enumerate(test_data):
    question = data["question"]
    print(f"\n[{i+1}/{len(test_data)}] {question}")

    try:
        response = rag.search(
            query_text=question,
            return_context=True
        )

        answer = response.answer
        cypher = response.retriever_result.metadata.get("cypher", "")
        context_text = convert_graph_context_to_text(response.retriever_result)

        results.append({
            "question": question,
            "answer": answer,
            "contexts": [context_text],  # RAGAS는 contexts를 list[str]로 받음
            "ground_truth": data["ground_truth"],
            "cypher": cypher,
        })

        print(f"  Cypher: {cypher}")
        print(f"  Context: {context_text}")
        print(f"  Answer: {answer}")

    except Exception as e:
        print(f"  오류 발생: {e}")
        results.append({
            "question": question,
            "answer": f"Error: {str(e)}",
            "contexts": ["No context retrieved due to error."],
            "ground_truth": data["ground_truth"],
            "cypher": "",
        })


[1/10] 매트릭스에서 누가 연기했나요?
  Cypher: MATCH (m:Movie) WHERE m.title CONTAINS 'Matrix' MATCH (p:Person)-[:ACTED_IN]->(m) RETURN m.title AS movie, p.name AS actor ORDER BY m.title, p.name
  Context: Executed Cypher query: MATCH (m:Movie) WHERE m.title CONTAINS 'Matrix' MATCH (p:Person)-[:ACTED_IN]->(m) RETURN m.title AS movie, p.name AS actor ORDER BY m.title, p.name
<Record movie='Matrix Reloaded, The' actor='Andy Arness'>
<Record movie='Matrix Reloaded, The' actor='Carrie-Anne Moss'>
<Record movie='Matrix Reloaded, The' actor='Christine Anu'>
<Record movie='Matrix Reloaded, The' actor='Keanu Reeves'>
<Record movie='Matrix Revolutions, The' actor='Helmut Bakaitis'>
<Record movie='Matrix Revolutions, The' actor='Kate Beahan'>
<Record movie='Matrix Revolutions, The' actor='Keanu Reeves'>
<Record movie='Matrix Revolutions, The' actor='Mary Alice'>
<Record movie='Matrix, The' actor='Carrie-Anne Moss'>
<Record movie='Matrix, The' actor='Hugo Weaving'>
<Record movie='Matrix, The' actor='Keanu Re

In [72]:
results_df = pd.DataFrame(results)
results_df[["question", "answer", "cypher"]].head(10)

,question,answer,cypher
0,매트릭스에서 누가 연기했나요?,"'매트릭스'에서 연기한 배우들은 Keanu Reeves, Carrie-Anne Mo...",MATCH (m:Movie) WHERE m.title CONTAINS 'Matrix...
1,매트릭스를 누가 감독했나요?,'매트릭스' 시리즈의 감독은 다음과 같습니다:\n\n- '매트릭스' (The Mat...,MATCH (m:Movie) WHERE m.title CONTAINS 'Matrix...
2,톰 행크스는 몇 편의 영화에 출연했나요?,톰 행크스는 43편의 영화에 출연했습니다.,cypher\nMATCH (p:Person)-[:ACTED_IN]->(m:Movie...
3,가장 높은 평점을 받은 상위 5개의 영화는 무엇인가요?,"1. Band of Brothers - 9.6\n2. Civil War, The -...",MATCH (m:Movie) WHERE m.imdbRating IS NOT NULL...
4,포레스트 검프는 어떤 장르에 속하나요?,"포레스트 검프는 코미디, 드라마, 로맨스, 전쟁 장르에 속합니다.",MATCH (m:Movie) WHERE m.title CONTAINS 'Forres...
5,매트릭스와 매트릭스 리로디드에 모두 출연한 배우는 누구인가요?,매트릭스와 매트릭스 리로디드에 모두 출연한 배우는 Carrie-Anne Moss와 ...,cypher\nMATCH (m1:Movie) WHERE m1.title CONTAI...
6,가장 많은 영화를 감독한 감독은 누구인가요?,Woody Allen은 42편의 영화를 감독했습니다.,cypher\nMATCH (d:Director)-[:DIRECTED]->(m:Mov...
7,1999년에 개봉한 영화는 무엇인가요?,"- 10 Things I Hate About You\n- 13th Warrior, ...",MATCH (m:Movie) WHERE m.year = 1999 RETURN m.t...
8,펄프 픽션에 출연한 배우는 누구인가요?,"펄프 픽션에 출연한 배우는 John Travolta, Laura Lovelace, ...",cypher\nMATCH (m:Movie) WHERE m.title CONTAINS...
9,인셉션과 장르가 비슷한 영화를 추천해주세요.,"- Hamlet\n- RoboCop\n- Getaway, The\n- Insomni...",cypher\nMATCH (m:Movie) WHERE m.title CONTAINS...


In [73]:
print(results_df["answer"][0])

'매트릭스'에서 연기한 배우들은 Keanu Reeves, Carrie-Anne Moss, Hugo Weaving, Laurence Fishburne입니다.


In [74]:
results_df.to_csv("graph_rag_results.csv", index=False)

## 7. RAGAS 평가 실행

In [75]:
from ragas import EvaluationDataset, SingleTurnSample

# RAGAS SingleTurnSample로 변환
samples = []
for r in results:
    sample = SingleTurnSample(
        user_input=r["question"],
        response=r["answer"],
        retrieved_contexts=r["contexts"],
        reference=r["ground_truth"],
    )
    samples.append(sample)

eval_dataset = EvaluationDataset(samples=samples)

In [76]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory
from ragas.metrics.collections import (
    Faithfulness,
    AnswerRelevancy,
    ContextRecall,
    ContextPrecision,
    AnswerAccuracy,
)

# RAGAS용 LLM & Embeddings 설정
client = AsyncOpenAI()
evaluator_llm = llm_factory(
    "gpt-4o",
    client=client,
    max_tokens=8192
)
evaluator_embeddings = embedding_factory("openai", model="text-embedding-3-small", client=client)

# 평가 지표 정의 (Collections API 사용)
metrics = {
    "faithfulness": Faithfulness(llm=evaluator_llm),
    "answer_relevancy": AnswerRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings),
    "context_recall": ContextRecall(llm=evaluator_llm),
    "context_precision": ContextPrecision(llm=evaluator_llm),
    "answer_accuracy": AnswerAccuracy(llm=evaluator_llm),
}

print("평가 지표:")
for name, metric in metrics.items():
    print(f"  - {name}")

평가 지표:
  - faithfulness
  - answer_relevancy
  - context_recall
  - context_precision
  - answer_accuracy


In [77]:
import asyncio

async def evaluate_sample(sample, idx, total):
    """단일 샘플에 대해 모든 지표를 평가"""
    print(f"\n[{idx+1}/{total}] 평가 중: {sample.user_input[:50]}...")

    result = {
        "user_input": sample.user_input,
        "response": sample.response,
        "reference": sample.reference,
    }

    # 각 지표별로 평가 - 각 지표에 맞는 파라미터만 전달
    for metric_name, metric in metrics.items():
        try:
            # 지표별로 필요한 파라미터만 전달
            if metric_name == "faithfulness":
                score_result = await metric.ascore(
                    user_input=sample.user_input,
                    response=sample.response,
                    retrieved_contexts=sample.retrieved_contexts
                )
            elif metric_name == "answer_relevancy":
                score_result = await metric.ascore(
                    user_input=sample.user_input,
                    response=sample.response
                )
            elif metric_name == "context_recall":
                score_result = await metric.ascore(
                    user_input=sample.user_input,
                    retrieved_contexts=sample.retrieved_contexts,
                    reference=sample.reference
                )
            elif metric_name == "context_precision":
                score_result = await metric.ascore(
                    user_input=sample.user_input,
                    reference=sample.reference,
                    retrieved_contexts=sample.retrieved_contexts
                )
            elif metric_name == "answer_accuracy":
                score_result = await metric.ascore(
                    user_input=sample.user_input,
                    response=sample.response,
                    reference=sample.reference
                )

            result[metric_name] = score_result.value
        except Exception as e:
            print(f"  {metric_name}: 오류 - {str(e)}")
            result[metric_name] = None

    return result

# 모든 샘플 평가
async def evaluate_all_samples():
    tasks = []
    for idx, sample in enumerate(samples):
        tasks.append(evaluate_sample(sample, idx, len(samples)))

    results = []
    for task in asyncio.as_completed(tasks):
        result = await task
        results.append(result)

    return results

# 평가 실행
print("=" * 50)
print("GraphRAG (Text2Cypher) 평가 시작")
print("=" * 50)

eval_results_list = await evaluate_all_samples()

# DataFrame으로 변환
eval_df = pd.DataFrame(eval_results_list)

print("\n" + "=" * 50)
print("GraphRAG (Text2Cypher) 평가 완료")
print("=" * 50)
print(f"\n총 {len(eval_df)}개 샘플 평가 완료")
print(f"평가 지표: {list(metrics.keys())}")

GraphRAG (Text2Cypher) 평가 시작

[4/10] 평가 중: 가장 높은 평점을 받은 상위 5개의 영화는 무엇인가요?...

[6/10] 평가 중: 매트릭스와 매트릭스 리로디드에 모두 출연한 배우는 누구인가요?...

[10/10] 평가 중: 인셉션과 장르가 비슷한 영화를 추천해주세요....

[8/10] 평가 중: 1999년에 개봉한 영화는 무엇인가요?...

[7/10] 평가 중: 가장 많은 영화를 감독한 감독은 누구인가요?...

[1/10] 평가 중: 매트릭스에서 누가 연기했나요?...

[3/10] 평가 중: 톰 행크스는 몇 편의 영화에 출연했나요?...

[5/10] 평가 중: 포레스트 검프는 어떤 장르에 속하나요?...

[2/10] 평가 중: 매트릭스를 누가 감독했나요?...

[9/10] 평가 중: 펄프 픽션에 출연한 배우는 누구인가요?...

GraphRAG (Text2Cypher) 평가 완료

총 10개 샘플 평가 완료
평가 지표: ['faithfulness', 'answer_relevancy', 'context_recall', 'context_precision', 'answer_accuracy']


## 8. 결과 분석

In [78]:
eval_df

,user_input,response,reference,faithfulness,answer_relevancy,context_recall,context_precision,answer_accuracy
0,톰 행크스는 몇 편의 영화에 출연했나요?,톰 행크스는 43편의 영화에 출연했습니다.,톰 행크스는 38편의 영화에 출연했습니다.,0.0,0.355771,0.0,0.0,0.25
1,가장 높은 평점을 받은 상위 5개의 영화는 무엇인가요?,"1. Band of Brothers - 9.6\n2. Civil War, The -...",가장 높은 평점을 받은 상위 5개의 영화는 다음과 같습니다: Band of Brot...,1.0,0.514437,1.0,1.0,1.00
2,매트릭스와 매트릭스 리로디드에 모두 출연한 배우는 누구인가요?,매트릭스와 매트릭스 리로디드에 모두 출연한 배우는 Carrie-Anne Moss와 ...,매트릭스와 매트릭스 리로디드에 모두 출연한 배우는 다음과 같습니다: Carrie-A...,1.0,0.243876,1.0,1.0,1.00
3,1999년에 개봉한 영화는 무엇인가요?,"- 10 Things I Hate About You\n- 13th Warrior, ...",1999년에 개봉한 영화는 다음과 같습니다: 10 Things I Hate Abou...,1.0,0.288254,0.0,1.0,0.50
4,포레스트 검프는 어떤 장르에 속하나요?,"포레스트 검프는 코미디, 드라마, 로맨스, 전쟁 장르에 속합니다.",포레스트 검프 영화와 장르는 다음과 같습니다: Forrest Gump - Comed...,1.0,0.986711,1.0,1.0,1.00
5,가장 많은 영화를 감독한 감독은 누구인가요?,Woody Allen은 42편의 영화를 감독했습니다.,가장 많은 영화를 감독한 감독은 Woody Allen - 42입니다.,1.0,0.365725,1.0,1.0,1.00
6,펄프 픽션에 출연한 배우는 누구인가요?,"펄프 픽션에 출연한 배우는 John Travolta, Laura Lovelace, ...",펄프 픽션 영화와 출연 배우는 다음과 같습니다: Pulp Fiction - John...,1.0,0.260557,1.0,1.0,1.00
7,매트릭스에서 누가 연기했나요?,"'매트릭스'에서 연기한 배우들은 Keanu Reeves, Carrie-Anne Mo...",매트릭스 시리즈 영화들과 출연 배우는 다음과 같습니다: Matrix Reloaded...,1.0,0.200616,1.0,1.0,0.50
8,매트릭스를 누가 감독했나요?,'매트릭스' 시리즈의 감독은 다음과 같습니다:\n\n- '매트릭스' (The Mat...,"매트릭스 시리즈 영화들과 감독은 다음과 같습니다: Matrix Reloaded, T...",1.0,0.269327,1.0,1.0,1.00
9,인셉션과 장르가 비슷한 영화를 추천해주세요.,"- Hamlet\n- RoboCop\n- Getaway, The\n- Insomni...",인셉션과 장르가 비슷한 영화는 다음과 같습니다: RoboCop - 9; Hamlet...,1.0,0.361459,1.0,1.0,0.50


In [79]:
eval_df.to_csv("graph_rag_evaluation_results_{}.csv".format(pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")), index=False)

In [80]:
# 지표별 평균 점수 요약
metric_cols = ["faithfulness", "answer_relevancy", "context_recall", "context_precision", "answer_accuracy"]
available_cols = [c for c in metric_cols if c in eval_df.columns]

summary = eval_df[available_cols].mean().round(4)
print("=" * 50)
print("GraphRAG 평가 지표 평균")
print("=" * 50)
for metric, score in summary.items():
    print(f"  {metric:25s}: {score:.4f}")
print("=" * 50)

GraphRAG 평가 지표 평균
  faithfulness             : 0.9000
  answer_relevancy         : 0.3847
  context_recall           : 0.8000
  context_precision        : 0.9000
  answer_accuracy          : 0.7750


In [81]:
# 점수가 낮은 질문 확인 (개선 포인트 파악)
if "answer_accuracy" in eval_df.columns:
    low_score = eval_df.nsmallest(3, "answer_accuracy")[["user_input", "answer_accuracy"]]
    print("Answer Accuracy가 낮은 질문 Top 3:")
    for _, row in low_score.iterrows():
        print(f"  [{row['answer_accuracy']:.2f}] {row['user_input']}")

Answer Accuracy가 낮은 질문 Top 3:
  [0.25] 톰 행크스는 몇 편의 영화에 출연했나요?
  [0.50] 1999년에 개봉한 영화는 무엇인가요?
  [0.50] 매트릭스에서 누가 연기했나요?


In [82]:
# Neo4j 연결 종료
driver.close()